In [0]:
import dlt
from pyspark.sql.functions import col, current_timestamp, regexp_extract

# Retrieve catalog from pipeline parameters, falling back safely to meridian_t12
catalog = spark.conf.get("catalog_name", spark.conf.get("catalog", "meridian_t12"))
volume_path = f"/Volumes/{catalog}/landing/raw_files"

entities = ["bookings", "payments", "properties", "destinations", "users", "hosts"]

def generate_bronze_table(entity_name):
    @dlt.table(
        name=f"bronze_{entity_name}",
        comment=f"Append-only raw layer for {entity_name}"
    )
    def create_streaming_table():
        return (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes", "true")
            # Store schema tracking inside the managed volume to avoid UC table path collision:
            .option("cloudFiles.schemaLocation", f"{volume_path}/_checkpoints/{entity_name}")
            .load(f"{volume_path}/{entity_name}/")
            .select(
                "*",
                col("_metadata.file_path").alias("source_file_path"),
                current_timestamp().alias("ingestion_time"),
                regexp_extract("_metadata.file_path", r"batch_(\d+)", 1).cast("integer").alias("batch_number")
            )
        )

# Register tables
for entity in entities:
    generate_bronze_table(entity)